In [1]:
import pandas as pd

df_ground_truth = pd.read_csv('data/ground_truth.csv')
ground_truth = df_ground_truth.to_dict(orient='records')
ground_truth[0]

{'question': 'I just found this course late — can I still sign up and follow along, or is it too late?',
 'document': '74eb249bbf'}

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
documents_llm = []

[documents_llm.append(doc) for doc in documents if doc['course'] == 'llm-zoomcamp']

[{'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 471}, {'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 253}, {'course': 'stock-markets-analytics-zoomcamp', 'course_name': 'Stock Markets Analytics Zoomcamp', 'path': '/json/stock-markets-analytics-zoomcamp.json', 'questions_count': 93}, {'course': 'ai-dev-tools-zoomcamp', 'course_name': 'AI Dev Tools Zoomcamp', 'path': '/json/ai-dev-tools-zoomcamp.json', 'questions_count': 41}, {'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 404}, {'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 118}]


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [3]:
documents = documents_llm

index = build_index(documents)

In [4]:
doc_idx = {}
for doc in documents:
  doc_idx[doc['id']] = doc

doc_idx[ground_truth[0]['document']]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [5]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()



In [6]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(index, client, course='llm-zoomcamp')

In [7]:
q = ground_truth[11]['question']
q

'Where can I find the stream link or video URL before an office hours session starts?'

In [8]:
answer = assistant.rag(q)

In [9]:
print(answer)
print(assistant.total_cost())

The video URL should be posted in the announcements channel on Telegram and Slack before the session begins. You can also watch it live on the DataTalksClub YouTube Channel.
0.00080475


In [10]:
from IPython.display import Markdown, display
display(Markdown(answer))

The video URL should be posted in the announcements channel on Telegram and Slack before the session begins. You can also watch it live on the DataTalksClub YouTube Channel.

In [11]:
# collect all the rag answers
doc = ground_truth[11]
doc_id = doc['document']
original_doc = doc_idx[doc_id]
answer_orig = original_doc['answer']
answer_orig

'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'

In [12]:
rag_result = {
  "question": doc['question'],
  "answer_orig": answer_orig,
  "answer_rag": answer,
  "document": doc_id
}
rag_result


{'question': 'Where can I find the stream link or video URL before an office hours session starts?',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'answer_rag': 'The video URL should be posted in the announcements channel on Telegram and Slack before the session begins. You can also watch it live on the DataTalksClub YouTube Channel.',
 'document': '489dd1c9d9'}

In [13]:
def generate_rag_answer(doc):
  question = doc['question']
  doc_id = doc['document']
  original_doc = doc_idx[doc_id]
  answer_orig = original_doc['answer']
  answer = assistant.rag(question)
  return {
    "question": question,
    "answer_orig": answer_orig,
    "answer_rag": answer,
    "document": doc_id
  }

assistant.reset_usage()


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

# with ThreadPoolExecutor(max_workers=10) as pool:
#   rag_results = map_progress(pool, ground_truth, generate_rag_answer)


  0%|          | 0/590 [00:00<?, ?it/s]

In [ ]:
# df_results = pd.DataFrame(rag_results)
# df_results.to_csv('data/rag_results.csv', index=False)

In [14]:
# LLM as a judge

df_answers = pd.read_csv('data/rag-answers-new.csv')
answers = df_answers.to_dict(orient="records")


In [15]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvalutation(BaseModel):
  reasoning: str = Field(description="Reasonging about the quality of the answer")
  score: Literal["good", "bad"] = Field(description="'good' if the answer is correct and complete, 'bad' if the answer is incorrect")

In [16]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [17]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [18]:
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

In [ ]:
rec = answers[0]
prompt = aqa_judge_prompt.format(
  question=rec['question'],
  answer_orig=rec['answer_orig'],
  answer_llm=rec['answer_llm']
)
prompt

'Question:\nIs it okay to join the course late if I just found it now?\n\nOriginal Answer (ground truth):\nYes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nAI Answer:\nYes, you can still join the course late. If you want a certificate, though, you need to submit your project while submissions are still being accepted.'

In [22]:
eval_result, usage = llm_structured_retry(
  client,
  aqa_judge_instructions,
  prompt,
  AnswerEvalutation,
)

In [25]:
display(Markdown(eval_result.reasoning))
print(eval_result.score)

The AI answer preserves the original meaning: late joining is allowed, and certificate eligibility depends on submitting the project before submissions close. This is semantically equivalent to the ground truth.

good


In [28]:
def evaluate_answer(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
  prompt = aqa_judge_prompt.format(
    question=question,
    answer_orig=answer_orig,
    answer_llm=answer_llm
  )
  
  eval_result, usage = llm_structured_retry(
    client,
    aqa_judge_instructions,
    prompt,
    AnswerEvalutation,
  )
  
  return eval_result, usage

In [34]:
def judge_record(rec): 
  eval_result, usage = evaluate_answer(
    rec['question'],
    rec['answer_orig'],
    rec['answer_llm']
  )
  result = {
    "question": rec['question'],
    "document": rec['document'],
    "score": eval_result.score,
    "reasoning": eval_result.reasoning
  }

  return result, usage


In [32]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=10) as pool:
  llm_judge = map_progress(pool, answers, judge_record)



  0%|          | 0/395 [00:00<?, ?it/s]

In [33]:
llm_judge[0]

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf',
 'score': 'good',
 'reasoning': 'The AI answer preserves the full meaning of the ground truth: late joining is allowed, and certificate eligibility depends on submitting the project before submissions close. This is semantically equivalent.'}

In [52]:
llm_judge_fake_usage = zip(llm_judge, [0] * len(llm_judge))

evalutations = []
usages = []
for result, usage in llm_judge_fake_usage:
  evalutations.append(result)
  usages.append(usage)

df_eval = pd.DataFrame(evalutations)


In [54]:
df_eval.score.value_counts()

score
good    381
bad      14
Name: count, dtype: int64

In [55]:
df_eval.score.value_counts(normalize=True)

score
good    0.964557
bad     0.035443
Name: proportion, dtype: float64

In [65]:
df_eval[df_eval.score == 'bad'].head()

,question,document,score,reasoning
15,How do the free GPU hours work on these cloud ...,c6c2888275,bad,The AI answer does not answer the question or ...
29,Is peer-review of the capstone project require...,69d122f12e,bad,The AI answer states that peer-reviewing 3 cap...
38,How will I know when a module is actually read...,96286b4be4,bad,The AI answer does not convey the ground truth...
106,Do I need an OpenAI API key just to check how ...,fe8fed31e6,bad,The ground truth says an OpenAI API key is not...
124,What model or provider should I switch to if I...,6e1d8a7b29,bad,The AI answer does not answer the question at ...


In [66]:
df_eval.to_csv('data/llm-judge-results.csv', index=False)

In [ ]:
# evaluation of the agent